# 03 · Graph construction on the recorded atlas: diagnostics and the pre-specified comparison

Runs on the analysis atlas you recorded in notebook 04 (`--atlas selected`; it stops if none is recorded); k scales
with that atlas's ROI count (k = round(f (R - 1)): 7 / 13 / 20 for 68 ROIs, 2 / 3 / 5 for 18).

1. **Diagnostics** (label-blind, development participants only): density, degree, isolated nodes, connected
   components, retained connectivity weight, within- vs between-participant edge stability, for every construction
   of `configs/final/graphs.yaml` (dense reference, union / mutual kNN with k = 10 / 20 / 30 % of the ROIs,
   proportional threshold, cortical contiguity, feature similarity).
2. **Comparison** (development CV only; the holdout is not run here): the reference models
   `exp4_band_mean` `plv__psd`, `exp5_multibranch` `plv__separate__mean` for every construction, both inverses, mode
   B.
3. **Selection** by the rule of `configs/final/selection.yaml`: a construction replaces `dense` only if it is better
   by at least 0.02 mean participant AUC AND in at least 75 %
   of the cells; otherwise `dense` is kept. The feature-similarity graph is reported, never selected. The decision is
   recorded once (`FINAL_v1/decisions/graph_construction.json`, with its atlas) and used by Exp 2-8
   (`--graph selected`). It is a pre-specified exploratory decision rule on development CV, not a test of
   statistical superiority: read the per-cell AUCs with their confidence intervals in the result tables.

**Holdout policy.** The 13-participant holdout was inspected during earlier experiments (LX / WX suites), so it is no longer an untouched test set. In this release it is evaluated once per final configuration and reported descriptively; it is never used to select architectures, graph constructions, k, atlas resolution, QC thresholds or any other setting (selection reads development CV results only).

GPU runtime (T4). Resumable.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move the release code pin deliberately to the newest commit
                      # (<outputs>/FINAL_v1/CODE_COMMIT.txt; unfinished work is then recomputed).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)

## 1. Diagnostics

In [ ]:
for inverse in ('lcmv', 'wmne'):
    sh(f"python scripts/final.py graphs --inverse {inverse} --mode B --atlas selected")

## 2. Comparison runs (development CV only)

In [ ]:
CONSTRUCTIONS = ["dense", "knn_union_f10", "knn_union_f20", "knn_union_f30", "knn_mutual_f10", "knn_mutual_f20", "knn_mutual_f30", "proportional_f20", "contiguity", "contiguity_knn_union_f20", "feature_similarity_knn_union_f20"]
REFERENCE = [["exp4_band_mean", "plv__psd"], ["exp5_multibranch", "plv__separate__mean"]]
ERRORS = []
for graph in CONSTRUCTIONS:
    for inverse in ["lcmv", "wmne"]:
        for exp, variant in REFERENCE:
            try:
                sh(f"python scripts/final.py run --exp {exp.split('_')[0]} --inverse {inverse} --mode B "
                   f"--atlas selected --graph {graph} --variants {variant}")
            except RuntimeError:
                ERRORS.append((graph, inverse, exp, variant))

## 3. Selection (development CV only)

In [ ]:
sh("python scripts/final.py report")
if ERRORS:
    raise RuntimeError(f'units did not complete: {ERRORS}; completed units are saved. Run all again.')
sh("python scripts/final.py select --atlas selected")